# Using AutoTokenizer and AutoModel to Get Embeddings

In [8]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr
import numpy as np

In [9]:
# Global device and model initialization for performance
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-mpnet-base-v2").to(DEVICE)

In [10]:

print(model)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [11]:
def embed_texts(texts: list[str]) -> np.ndarray:
    """
    Generates embeddings for a list of texts in an optimized, batch-aware manner.

    Optimizations:
    1. Processes a batch (list) of texts at once.
    2. Uses torch.float16 (half-precision) for faster GPU calculation.
    3. Manages tensors on the detected DEVICE (CUDA or CPU).

    Args:
        texts: A list of strings to embed.

    Returns:
        A NumPy array where each row is the embedding vector for a corresponding text.
    """
    if not tokenizer or not model:
        raise RuntimeError("Model and tokenizer must be initialized before calling embed_texts.")

    # 1. Tokenization and Device Transfer (Batch Processing)
    inputs = tokenizer(
        texts, 
        return_tensors="pt", 
        truncation=True, 
        padding=True,
        max_length=512 # Good practice to specify max length
    ).to(DEVICE)
    
    # 2. Model Inference
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Mean Pooling and Conversion to NumPy
    # Use mean pooling of the last hidden state over the sequence length dimension (dim=1)
    # The .cpu().numpy() moves the result back to the CPU memory and converts it for standard use.
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    
    return embeddings


In [12]:
# less than 512 tokens per sentence
sentences1 = ["I love dogs", "He is driving a car", "AI is amazing", "The sentence transformers are great!", "you are dead.", "The dog is fast."]
sentences2 = ["I adore dogs", "He drives a vehicle", "Artificial Intelligence is great", "The Large Language mode1s are fantastic.", "you are alive.", 
              "The dog is Not fast."]

In [13]:
# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

print(type(one), one.shape)  # Should print: <class 'numpy.ndarray'> (5, embedding_dim)
print(type(two), two.shape)  # Should print: <class 'numpy.ndarray'> (5, embedding_dim)

<class 'numpy.ndarray'> (6, 768)
<class 'numpy.ndarray'> (6, 768)


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between embeddings
similarity = cosine_similarity(one, two)

similarity.diagonal()

array([0.94789964, 0.72262335, 0.7956592 , 0.21364246, 0.64911276,
       0.9377249 ], dtype=float32)

# Dot product between two embeddings
- we should get same values

In [15]:
# Compute row norms
one_norms = np.linalg.norm(one, axis=1, keepdims=True)
two_norms = np.linalg.norm(two, axis=1, keepdims=True)

# Normalize rows in-place by dividing each element by the row norm
one /= one_norms
two /= two_norms

cosine_similarities = np.sum(one * two, axis=1)
cosine_similarities = np.dot(one,two.T).diagonal()

cosine_similarities

array([0.9478998 , 0.7226232 , 0.7956592 , 0.21364243, 0.64911276,
       0.93772495], dtype=float32)

# Different Model

In [16]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/nli-mpnet-base-v2")
model = AutoModel.from_pretrained("sentence-transformers/nli-mpnet-base-v2").to(DEVICE)

# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

# Compute row norms
one_norms = np.linalg.norm(one, axis=1, keepdims=True)
two_norms = np.linalg.norm(two, axis=1, keepdims=True)

# Normalize rows in-place by dividing each element by the row norm
one /= one_norms
two /= two_norms

cosine_similarities = np.sum(one * two, axis=1)
cosine_similarities = np.dot(one,two.T).diagonal()

cosine_similarities

array([0.9227279 , 0.8077708 , 0.48517326, 0.27817354, 0.33582848,
       0.8128757 ], dtype=float32)

In [17]:
# Using PyTorch for cosine similarity
one_tensor = torch.tensor(one).to(DEVICE)
two_tensor = torch.tensor(two).to(DEVICE)


cosine_similarities = F.cosine_similarity(one_tensor, two_tensor).to('cpu').numpy()
cosine_similarities

array([0.92272806, 0.80777085, 0.48517329, 0.2781734 , 0.3358286 ,
       0.81287587], dtype=float32)

Spearman’s rank correlation coefficient (denoted as $\rho$ or $r_s$) is often used to evaluate the agreement between cosine similarity scores from vector embeddings and human-labeled similarity scores.

In [18]:
from scipy.stats import spearmanr

# Example cosine similarity scores and human-labeled scores
human_scores = [5, 4, 4, 2 ,0, 0] 

correlation, p_value = spearmanr(cosine_similarities, human_scores)
print(f"Spearman's rank correlation: {correlation} & {p_value}")

Spearman's rank correlation: 0.44136741475237473 & 0.3809392104771887


#### Cosine similarity can range from -1 to 1, but the human scores are 0–5. We can scale the cosine scores to the same range:

In [19]:
from sklearn.preprocessing import MinMaxScaler

# Convert cosine similarities to a NumPy array
cosine_similarities = np.array(cosine_similarities).reshape(-1, 1)

# Scale to range [0, 5] to match human scores
scaler = MinMaxScaler(feature_range=(0, 5))
cosine_scaled = scaler.fit_transform(cosine_similarities).flatten()

print(f"Scaled cosine similarities: {cosine_scaled}")

correlation, p_value = spearmanr(cosine_scaled, human_scores)
print(f"\nSpearman's rank correlation after scaling: {correlation:.4f} & p-value: {p_value:.4f}")

Scaled cosine similarities: [5.         4.108243   1.6057589  0.         0.44724846 4.1478443 ]

Spearman's rank correlation after scaling: 0.4414 & p-value: 0.3809


#### Sometimes, small differences in embeddings dominate the raw cosine similarity. You can compress extreme values to reduce noise:

In [20]:
cosine_smoothed = 5 / (1 + np.exp(-10 * (cosine_scaled/5 - 0.5)))  # Sigmoid to [0,5]
correlation_smoothed, p_value = spearmanr(cosine_smoothed, human_scores)
print(f"Spearman's rank correlation after sigmoid smoothing: {correlation_smoothed:.4f} & p-value: {p_value:.4f}")

Spearman's rank correlation after sigmoid smoothing: 0.4414 & p-value: 0.3809


# Smaller embedding Model (all-MiniLM-L6-v2)

In [21]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(DEVICE)

In [22]:
# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

print(type(one), one.shape)  # Should print: <class 'numpy.ndarray'> (5, embedding_dim)
print(type(two), two.shape)  # Should print: <class 'numpy.ndarray'> (5, embedding_dim)

<class 'numpy.ndarray'> (6, 384)
<class 'numpy.ndarray'> (6, 384)


In [23]:
# Using PyTorch for cosine similarity
one_tensor = torch.tensor(one).to(DEVICE)
two_tensor = torch.tensor(two).to(DEVICE)


cosine_similarities = F.cosine_similarity(one_tensor, two_tensor).to('cpu').numpy()
cosine_similarities

array([0.8826928 , 0.75069606, 0.76624835, 0.33188027, 0.8151611 ,
       0.9363439 ], dtype=float32)

In [24]:
# Example cosine similarity scores and human-labeled scores
human_scores = [5, 4, 4, 2 ,0, 0] 

correlation, p_value = spearmanr(cosine_similarities, human_scores)
print(f"Spearman's rank correlation: {correlation} & {p_value}")

Spearman's rank correlation: -0.1765469659009499 & 0.7379309324353432


In [25]:
from sentence_transformers import SentenceTransformer, util

# Input sentences
s1 = "The dog is fast."
s2 = "The dog is not fast."

# Load model and move to DEVICE
# model = SentenceTransformer('all-mpnet-base-v2').to(DEVICE)
model = SentenceTransformer('nli-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)
# model = SentenceTransformer('sentence-t5-large').to(DEVICE)

# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:nli-mpnet-base-v2'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")

--- Model:nli-mpnet-base-v2'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.81164283


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
# Load model and move to DEVICE
model = SentenceTransformer('all-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('nli-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)
# model = SentenceTransformer('sentence-t5-large').to(DEVICE)

# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:all-mpnet-base-v2'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")

--- Model:all-mpnet-base-v2'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.93956923


In [27]:
from sentence_transformers import SentenceTransformer, util

# Input sentences
s1 = "The dog is fast."
s2 = "The dog is not fast."

# Load model and move to DEVICE
# model = SentenceTransformer('all-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('nli-mpnet-base-v2').to(DEVICE)
model = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)
# model = SentenceTransformer('sentence-t5-large').to(DEVICE)

# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:all-MiniLM-L6-v2'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")

--- Model:all-MiniLM-L6-v2'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.96619821


In [28]:
# Load model and move to DEVICE
# model = SentenceTransformer('all-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('nli-mpnet-base-v2').to(DEVICE)
# model = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)
model = SentenceTransformer('sentence-t5-large').to(DEVICE)

# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:sentence-t5-large'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")

--- Model:sentence-t5-large'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.91827089


- Larger models like 'sentence-t5-large' were unable to differentiate between two embeddings like smaller models.

In [29]:
from sentence_transformers import CrossEncoder


# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder('cross-encoder/stsb-roberta-base').to(DEVICE)

# Predict similarity score
similarity = model.predict([(s1, s2)])[0]

print(f"--- Model: cross-encoder/stsb-roberta-base ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity:.4f}")


--- Model: cross-encoder/stsb-roberta-base ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.2924


In [30]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder('cross-encoder/nli-distilroberta-base').to(DEVICE)

# Predict logits (1D tensor)
logits = model.predict([(s1, s2)], convert_to_tensor=True)  # shape: [3]

# Apply softmax along dim=0
probs = F.softmax(logits, dim=1)  # still a 1D tensor

labels = ["entailment", "neutral", "contradiction"]

# Iterate correctly: convert to list of floats
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.4f}")

entailment     : 0.9907
neutral        : 0.0050
contradiction  : 0.0043


In [31]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder('cross-encoder/nli-deberta-v3-small').to(DEVICE)

# Predict logits (1D tensor)
logits = model.predict([(s1, s2)], convert_to_tensor=True)  # shape: [3]

# Apply softmax along dim=0
probs = F.softmax(logits, dim=1)  # still a 1D tensor

labels = ["entailment", "neutral", "contradiction"]

# Iterate correctly: convert to list of floats
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.4f}")


entailment     : 0.9965
neutral        : 0.0012
contradiction  : 0.0023


In [32]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder('cross-encoder/stsb-roberta-large').to(DEVICE)

# Predict similarity score
similarity = model.predict([(s1, s2)])

print(f"--- Model: cross-encoder/stsb-roberta-large ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity[0]:.6f}")

--- Model: cross-encoder/stsb-roberta-large ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.463631


In [33]:
model = SentenceTransformer('stsb-roberta-large').to(DEVICE)

s1_emb = model.encode([s1], convert_to_tensor=True)
s2_emb = model.encode([s2], convert_to_tensor=True)

similarity = F.cosine_similarity(s1_emb, s2_emb)

print(f"--- Model: stsb-roberta-large ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity.item():.6f}")
print(f"\nEmbedding Shape: {s1_emb.shape}")

--- Model: stsb-roberta-large ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.756703

Embedding Shape: torch.Size([1, 1024])


In [34]:
model = SentenceTransformer('roberta-large-nli-stsb-mean-tokens').to(DEVICE)

s1_emb = model.encode([s1], convert_to_tensor=True)
s2_emb = model.encode([s2], convert_to_tensor=True)

similarity = F.cosine_similarity(s1_emb, s2_emb)

print(f"--- Model: roberta-large-nli-stsb-mean-tokens ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity.item():.6f}")
print(f"\nEmbedding Shape: {s1_emb.shape}")

--- Model: roberta-large-nli-stsb-mean-tokens ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.756703

Embedding Shape: torch.Size([1, 1024])


In [35]:
model = SentenceTransformer('roberta-base-nli-stsb-mean-tokens').to(DEVICE)

s1_emb = model.encode([s1], convert_to_tensor=True)
s2_emb = model.encode([s2], convert_to_tensor=True)

similarity = F.cosine_similarity(s1_emb, s2_emb)

print(f"--- Model: roberta-base-nli-stsb-mean-tokens ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity.item():.6f}")
print(f"\nEmbedding Shape: {s1_emb.shape}")

--- Model: roberta-base-nli-stsb-mean-tokens ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.764653

Embedding Shape: torch.Size([1, 768])


In [36]:
# # Entailment vs Contradiction vs Neutral

# # Load a cross-encoder that directly outputs similarity scores
# model = CrossEncoder('cross-encoder/nli-roberta-base').to(DEVICE)

# # Predict logits (3 values: entailment, neutral, contradiction)
# logits = model.predict([(s1, s2)], convert_to_tensor=True)

# # Apply softmax to get probabilities
# probs = F.softmax(logits, dim=1)

# # Map to labels
# labels = ["entailment", "neutral", "contradiction"]

# # Show results
# for label, prob in zip(labels, probs[0]):
#     print(f"{label:15s}: {prob.item():.4f}")

In [37]:
# Define sentences
s1 = "The dog is very fast."
s2 = "The dog is not fast."

In [38]:
# Use a stronger NLI model trained on negation-sensitive data
model = CrossEncoder('cross-encoder/nli-deberta-v3-small').to(DEVICE)

# Predict logits (3 values: entailment, neutral, contradiction)
logits = model.predict([(s1, s2)], convert_to_tensor=True)

# Apply softmax to get probabilities
probs = F.softmax(logits, dim=1)

# Map to labels
labels = ["entailment", "neutral", "contradiction"]

# Show results
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.6f}")

entailment     : 0.997232
neutral        : 0.001059
contradiction  : 0.001709


In [39]:
# Initialize the CrossEncoder model
model = CrossEncoder('cross-encoder/nli-deberta-v3-base').to(DEVICE)

# Sentence pairs for NLI
sentence_pairs = [
    ("The dog is very fast."),
    ("The dog is not fast.")
]

# Predict logits for each pair
logits = model.predict(sentence_pairs)

# Apply softmax to get probabilities
probs = F.softmax(torch.from_numpy(logits), dim=0)

# Map to labels
labels = ["entailment", "neutral", "contradiction"]

# Show results
for label, prob in zip(labels, probs):
    print(f"{label:15s}: {prob.item():.6f}")

entailment     : 0.999285
neutral        : 0.000220
contradiction  : 0.000495


In [40]:
# Initialize the model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2').to(DEVICE)

# Compute embeddings
embeddings = model.encode([s1, s2])

# Calculate cosine similarity
cosine_sim = util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0].item()

# Output the similarity score
print(f"Cosine Similarity: {cosine_sim:.4f}")


Cosine Similarity: 0.8971


In [41]:
embeddings.shape

(2, 768)

## Choose an embedding Model

In [51]:
sentence_pairs = [
    ("What is the capital of France?", "Paris is the capital."),
    ("What is the capital of France?", "The Eiffel Tower is in Paris."),
    ("What is the capital of France?", "Berlin is a city in Germany.")
]



In [52]:
from sentence_transformers import CrossEncoder

# 1. Load the Cross-Encoder model
# This model will predict a score (0 to 1) for the semantic similarity of the pair.
reranker_model = CrossEncoder('cross-encoder/stsb-roberta-large').to(DEVICE)

# 2. Score the pairs
# The .predict() method processes the pairs and returns a list of scores.
similarity_scores = reranker_model.predict(sentence_pairs)

similarity_scores

array([0.3123381 , 0.13270384, 0.01069016], dtype=float32)

In [56]:
# Load a cross-encoder that directly outputs similarity scores
reranker_model = CrossEncoder('cross-encoder/stsb-roberta-base').to(DEVICE)

# Predict similarity score
similarity = reranker_model.predict(sentence_pairs)

similarity

array([0.26188636, 0.21285567, 0.1247337 ], dtype=float32)

In [45]:
model = SentenceTransformer('roberta-large-nli-stsb-mean-tokens').to(DEVICE)

s1_emb = model.encode([s1], convert_to_tensor=True)
s2_emb = model.encode([s2], convert_to_tensor=True)

similarity = F.cosine_similarity(s1_emb, s2_emb)

print(f"--- Model: roberta-large-nli-stsb-mean-tokens ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity.item():.6f}")
print(f"\nEmbedding Shape: {s1_emb.shape}")

--- Model: roberta-large-nli-stsb-mean-tokens ---
Sentence 1: 'The dog is very fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.557049

Embedding Shape: torch.Size([1, 1024])


In [58]:
model = SentenceTransformer('all-distilroberta-v1').to(DEVICE)

s1_emb = model.encode([s1], convert_to_tensor=True)
s2_emb = model.encode([s2], convert_to_tensor=True)

similarity = F.cosine_similarity(s1_emb, s2_emb)

print(f"--- Model: all-distilroberta-v1 ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Predicted Similarity Score: {similarity.item():.6f}")
print(f"\nEmbedding Shape: {s1_emb.shape}")

--- Model: all-distilroberta-v1 ---
Sentence 1: 'The dog is very fast.'
Sentence 2: 'The dog is not fast.'
Predicted Similarity Score: 0.834122

Embedding Shape: torch.Size([1, 768])
